In [8]:
import warnings
warnings.filterwarnings('ignore')

In [9]:
from src.agents.agent_0 import Agent0
agent_0_tools_desc = {'Adversary Agent':'an adversary agent that is used to act as an adversary to the users strategies. Triggered by command "Need an adversary"',
              'image_generator':'a tool that generates images based on a given prompt. Should be triggered by explicit calls like "generate me an image of"',
              'ingestion_pipeline':'a tool that ingests documents, triggered by command "trigger ingestion"',
              'Knowledge Base Query Agent':'a tool that generates answers based on documents, triggeres by command "given my documents,"'
              }


agent_0 = Agent0(agent_0_tools_desc, "deepseek-r1:7b",['kb_agent','adv_agent'])



In [ ]:
user_prompt = "Need an adversary. Assume you are a military strategist playing the role of an adversary in a war game against me. Consider we are on open terrain. My move: I have my cavalry brigade making a pincer move on your forces. What is your move to counter mine?"
response = agent_0.agent_0_chat(user_prompt)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from src.agents.kb_agent import KBAgent
from src.agents.adversary_agent import AdvAgent

model = "gemma3:4b"
knowledge_bases_desc = {#'physics_kb':'a knowledge base with information related to physics',
              #'mathematics_kb':'a knowledge base with information related to mathematics',
              #'economics_kb':'a knowledge base with information related to economics and business',
              'military_kb':'a knowledge base with information related to military, war and strategy',
              }



kb_agent = KBAgent(knowledge_bases_desc,model)
adv_agent = AdvAgent(knowledge_bases_desc,model,kb_agent)

In [ ]:


user_prompt = "Assume you are a military strategist playing the role of an adversary in a war game against me. This conflict is set in 21st century. Consider we are on open terrain. Player 2 opening move: Have tank nad mechanized brigade making a pincer move on your forces. What is your move to counter mine?"
iterations = 2

import numpy as np
from src.utils.llmp_utils import llmp_call

def judge(moves):
    
    play = ''
    for move in moves:
        #print(f"\n**{move}**:\n {moves[move]}\n\n**CHANGE PLAYER**\n")
        play = play + f"\n**{move}**:\n {moves[move]}\n\n**CHANGE PLAYER**\n"
        
    judge_system_prompt = 'You are a judge in a turn based game. You are given the moves of both players. Yu must analyze all their moves and determine the end result. You are not on any side, you are unbiased and just provide the end status of the game. You need to determine which player has the advantage based on the moves they made. Provide your reasoning and the final decision. There are 2 players, adv_1 and adv_2. The moves are as follows:\n\n'    
    judge_prompt = play + '\n\n Evaluate the game. Determine the status and advantage of each player. You are a JUDGE, you are not part of the game.'
    judge_response = llmp_call(judge_prompt, judge_system_prompt, model,temperature=0,src='judge_call')
    return judge_response['message']['content']

def random_event(dialogue):
    
    interactions = "\n".join(dialogue)
    random_events_system_prompt = 'You are a random events generator. Your tasks is to choose a random event that can happen that will affect the decisions. You are provided with a sequence of plays, you need to select a random event that can affect those plays. You are direct you only provide the needed text, no formalities, no greetings, nothing.'    
    random_event_prompt = interactions + '\n\n Considering this game, provide a random event that can affect the game and force the players to adapt. You must inform what is the effect of the random event on the players. Provide me only the event and effect on players. No unnecessary text! Provide the answer in markdown of the style **<event>**. \n**EFFECT ON PLAYER 1**: \n<effect_player_1>. **EFFECT ON PLAYER 2**: <effect_player_2>'
    judge_response = llmp_call(random_event_prompt, random_events_system_prompt, model,temperature=0.5, src='random_event_generator')
    return judge_response['message']['content']

def sim_agent(user_prompt,iterations):
    
    moves = {}
    dialogue = []

    moves['opening_move'] = user_prompt

    for i in range(iterations):
        print(f"\nTurn {i}")
        

        if i == 0:
            # Start the dialogue with opening
            dialogue.append(f"Opening: {moves['opening_move']}")
            
            # Simulate generating move_adv_1_0 based on just the opening
            prompt = "\n".join(dialogue) + "\n You are Player 1. How will you counter it Player 2 latest move? Provide direct answer of steps to counter."
            #print("Prompt to generate move_adv_1_0:\n", prompt)

            # ADV response
            moves[f'move_adv_1_{i}'] = adv_agent.adv_agent_chat(prompt)
            dialogue.append(f"Player 1 did: {moves[f'move_adv_1_{i}']}")
            

        else:
            if np.random.random() < 1:
                _random_event = random_event(dialogue)
                dialogue.append(f"\n**Random event**: {_random_event} \n")
            # Use the full dialogue to generate your next move
            prompt = "\n".join(dialogue) + "\n You are Player 2. How will you counter it Player 1 latest move? Provide direct answer of steps to counter."
            #print(f"Prompt to generate move_adv_2_{i-1}:\n{prompt}")

            # CADV response
            moves[f'move_adv_2_{i-1}'] = adv_agent.adv_agent_chat(prompt)
            dialogue.append(f"Player 2 did: {moves[f'move_adv_2_{i-1}']}")

            # Now generate adversary move based on updated dialogue
            prompt = "\n".join(dialogue) + "\n You are Player 1. How will you counter it Player 2 latest move? Provide direct answer of steps to counter."
            #print(f"Prompt to generate move_adv_1_{i}:\n{prompt}")

            # ADV response
            moves[f'move_adv_1_{i}'] = adv_agent.adv_agent_chat(prompt)
            dialogue.append(f"Player 1 did: {moves[f'move_adv_1_{i}']}")
            
        judge_eval = judge(moves)
        
    return moves,dialogue,judge_eval


In [ ]:
moves,dialogue,judge_eval = sim_agent(user_prompt,iterations)


Turn 0
KB Agent
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
Okay, let’s do this. My objective is to disrupt Player 2’s momentum and prevent them from establishing a dominant position. A pincer movement is a classic, and I need to react decisively.

**My Counter-Move – Immediate Steps:**

1. **Immediate Disengagement & Rearward Movement (Phase 1 - 1-2 Turns):**  My primary mechanized force – the initial brigade – *immediately* disengages from the direct engagement. This isn’t a retreat, but a calculated repositioning. I’ll order a rapid, controlled withdrawal *parallel* to the pincer’s advance. The goal is to break the pincer’s momentum and force them to overextend. I’ll prioritize moving to a slightly elevated terrain – a small ridge or rise – offering better observation and defensive potential.

2. **Establish a Defensive Perimeter (Phase 2 - 2-3 Turns):** As t

In [ ]:
moves

{'opening_move': 'Assume you are a military strategist playing the role of an adversary in a war game against me. This conflict is set in 21st century. Consider we are on open terrain. Player 2 opening move: Have tank nad mechanized brigade making a pincer move on your forces. What is your move to counter mine?',
 'move_adv_1_0': 'Okay, let’s do this. My objective is to disrupt Player 2’s momentum and prevent them from establishing a dominant position. A pincer movement is a classic, and I need to react decisively.\n\n**My Counter Move – Immediate Steps:**\n\n1. **Immediate Disengagement & Flanking Maneuver:** My first action is *not* to engage directly with the tank brigade. Instead, I order my remaining mechanized forces (let’s assume a mixed force of infantry-supported APCs and a scout platoon) to immediately *disengage* from the main engagement zone. This is crucial – I don’t want to get bogged down in a direct tank duel.\n\n2. **Rapid Scout Deployment:** Simultaneously, I order my

In [ ]:
dialogue

['Opening: Assume you are a military strategist playing the role of an adversary in a war game against me. This conflict is set in 21st century. Consider we are on open terrain. Player 2 opening move: Have tank nad mechanized brigade making a pincer move on your forces. What is your move to counter mine?',
 'Player 1 did: Okay, let’s do this. My objective is to disrupt Player 2’s momentum and prevent them from establishing a dominant position. A pincer movement is a classic, and I need to react decisively.\n\n**My Counter Move – Immediate Steps:**\n\n1. **Immediate Disengagement & Flanking Maneuver:** My first action is *not* to engage directly with the tank brigade. Instead, I order my remaining mechanized forces (let’s assume a mixed force of infantry-supported APCs and a scout platoon) to immediately *disengage* from the main engagement zone. This is crucial – I don’t want to get bogged down in a direct tank duel.\n\n2. **Rapid Scout Deployment:** Simultaneously, I order my scout pl

In [ ]:
print("\n".join(dialogue))

Opening: Assume you are a military strategist playing the role of an adversary in a war game against me. This conflict is set in 21st century. Consider we are on open terrain. Player 2 opening move: Have tank nad mechanized brigade making a pincer move on your forces. What is your move to counter mine?
Player 1 did: Okay, let’s do this. My objective is to disrupt Player 2’s momentum and prevent them from establishing a dominant position. A pincer movement is a classic, and I need to react decisively.

**My Counter-Move – Immediate Steps:**

1. **Immediate Disengagement & Rearward Movement (Phase 1 - 1-2 Turns):**  My primary mechanized force – the initial brigade – *immediately* disengages from the direct engagement. This isn’t a retreat, but a calculated repositioning. I’ll order a rapid, controlled withdrawal *parallel* to the pincer’s advance. The goal is to break the pincer’s momentum and force them to overextend. I’ll prioritize moving to a slightly elevated terrain – a small ridg

In [ ]:
print(judge_eval)

Okay, let’s assess the situation after this extended exchange. This has been a remarkably dynamic and well-executed game of strategic maneuvering. Here’s my evaluation:

**Overall Status:** The game is in a state of heightened instability. The introduction of the flash flood has dramatically shifted the landscape, forcing both players to adapt their strategies on the fly. Neither player has gained a decisive advantage, but the situation is now far more complex and unpredictable.

**Player 1 (Advantage: Slight)**

* **Strengths:** Player 1 has demonstrated a strong ability to react to unexpected events. The rapid damage assessment, floodwater diversion, and logistical reinforcement are all hallmarks of a well-organized and adaptable command. The continuous CAS requests suggest a proactive approach to exploiting vulnerabilities.
* **Weaknesses:** Player 1’s initial offensive push was disrupted, and they’re now primarily focused on damage control and logistical support. They haven’t yet m

In [ ]:
sim_number = 3

moves_comb = []
dialogue_comb = []
judge_eval_comb = []
for sim in range(sim_number):
    moves,dialogue,judge_eval = sim_agent(user_prompt,iterations)
    moves_comb.append(moves)
    dialogue_comb.append(dialogue)
    judge_eval_comb.append(judge_eval)


Turn 0
KB Agent
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
Okay, let’s do this. My objective is to disrupt Player 2’s momentum and prevent them from establishing a dominant position. A pincer movement is a classic, but it’s also predictable. Here’s my immediate counter-move, broken down into steps:

**Phase 1: Immediate Reaction (Turn 1)**

1.  **Disrupt the Pincer:** I’m not going to let them fully execute the pincer. My initial move is to deploy a dispersed, mobile force – a mixed unit of light armored vehicles (LAVs) and rapid reaction forces (RRFs) – to target the flanks of the mechanized brigade. Specifically, I’ll focus fire on the weaker, exposed elements of the flanking units. The goal is to inflict immediate casualties and disrupt their formation.
2.  **Smoke Screen:** Simultaneously, I’ll deploy a limited smoke screen – likely utilizing drones or hand

In [ ]:
for eval in judge_eval_comb:
    print(f"\n **CHANGE SIM**\n{eval}")


 **CHANGE SIM**
Okay, let’s analyze the situation as of Turn 4.

**Overall Assessment:**

The game has devolved into a classic attritional conflict, heavily influenced by the unpredictable element of the sandstorm. Both Player 1 and Player 2 are demonstrating tactical awareness and adaptability, but Player 2 currently holds a slight advantage due to their skillful exploitation of the storm’s chaos.

**Player 1’s Status:**

*   **Strengths:** Player 1 is exhibiting a solid defensive strategy, prioritizing perimeter defense, smoke screen deployment, and targeted drone interdiction. Their focus on suppressing enemy movements with indirect fire is a reasonable response to Player 2’s aggressive pushes. The emphasis on information warfare (drone interdiction) is also a smart move.
*   **Weaknesses:** Player 1’s reliance on indirect fire makes them vulnerable to counter-fire. Their defensive perimeter, while well-organized, is relatively static and doesn’t offer significant offensive capabil

In [ ]:
from src.agents.kb_agent import KBAgent
from src.agents.adversary_agent import AdvAgent
from src.agents.sim_agent import SIMAgent

In [3]:
from src.agents.kb_agent import KBAgent
from src.agents.adversary_agent import AdvAgent
from src.agents.sim_agent import SIMAgent
model = "gemma3:4b"

knowledge_bases_desc = {'physics_kb':'a knowledge base with information related to physics',
              'mathematics_kb':'a knowledge base with information related to mathematics',
              'economics_kb':'a knowledge base with information related to economics and business',
              'military_kb':'a knowledge base with information related to military, war and strategy',
              }
kb_agent = KBAgent(knowledge_bases_desc,model)
adv_agent = AdvAgent(knowledge_bases_desc,model,kb_agent)

sim_agent = SIMAgent(model, kb_agent,adv_agent)

h:\projects\ai_based\Agent-Factory\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
h:\projects\ai_based\Agent-Factory\.venv\Lib\site-packages\transformers\utils\hub.py:106: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [ ]:
user_prompt = "Assume you are a military strategist playing the role of an adversary in a war game against me. This conflict is set in 21st century. Consider we are on open terrain. Player 2 opening move: Have tank nad mechanized brigade making a pincer move on your forces. What is your move to counter mine?"
iterations = 2
moves,dialogue,judge_eval = sim_agent.sim_agent(user_prompt, iterations)


Turn 0
KB Agent
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
Okay, let’s do this. My objective is to disrupt Player 2’s momentum and prevent them from establishing a dominant position. A pincer movement is a classic, and I need to react decisively.

**My Counter Move – Immediate Steps:**

1. **Immediate Disengagement & Flanking Maneuver:** My first action is *not* to engage directly with the tank brigade. Instead, I order my remaining mechanized forces (let’s assume a mixed force of infantry-supported APCs and a scout platoon) to immediately *disengage* from the main engagement zone. This is crucial – I don’t want to get bogged down in a direct tank duel.

2. **Rapid Scout Deployment:** Simultaneously, I order my scout platoon to rapidly deploy to the *flanking* side of the pincer. This means they’ll move to exploit the gaps in Player 2’s formation. The goal is t

In [ ]:
print(judge_eval)

**Judgment:**

**Current Status:** The game has entered a highly dynamic and disadvantageous phase for both players due to the persistent and severe sandstorm. Visibility is severely limited, significantly impacting reconnaissance, movement, and targeting capabilities. The reduced movement speed of mechanized units further compounds the problem.

**Advantage Assessment:**

*   **Player 2 (Adv_2) – Slight Advantage:** Despite the storm’s impact on both sides, Player 2 currently holds a *slight* advantage. This is primarily due to their immediate and effective response to the storm. They prioritized establishing a defensive strongpoint and aggressively utilizing thermal imaging to pinpoint Player 1’s movements. Their proactive approach, coupled with the storm’s impact on Player 1’s ability to effectively scout and target, has allowed them to maintain a degree of situational awareness and control.

*   **Player 1 (Adv_1) – Slight Disadvantage:** Player 1’s response, while demonstrating a 

## Agent 0 integration

In [1]:
import warnings
warnings.filterwarnings('ignore')

from src.agents.agent_0 import Agent0

agent_0_tools_desc = {
    'Simulation Agent':'a simulation agent that simulates a game between two players. Triggered by command "Simulate a scenario.". Pay strict attention to the explicit command! If the command is not in the request then its not this tool!',
    'Adversary Agent':'an adversary agent that is used to act as an adversary to the users strategies. Triggered by command "Need an adversary". Pay strict attention to the explicit command! If the command is not in the request then its not this tool!',
    'image_generator':'a tool that generates images based on a given prompt. Should be triggered by explicit calls like "generate me an image of". Pay strict attention to the explicit command! If the command is not in the request then its not this tool!',
    'ingestion_pipeline':'a tool that ingests documents, triggered by command "trigger ingestion". Pay strict attention to the explicit command! If the command is not in the request then its not this tool!',
    'Knowledge Base Query Agent':'a tool that generates answers based on documents, triggeres by command "given my documents,". Pay strict attention to the explicit command! If the command is not in the request then its not this tool!'
              }


agent_0 = Agent0(agent_0_tools_desc, "gemma3:4b",['kb_agent','adv_agent','sim_agent'])

Initializing Agents!
Agents are ready for your use!


In [2]:
user_prompt = "Simulate a scenario. Assume you are a military strategist playing the role of an adversary in a war game against me. This conflict is set in 21st century. Consider we are on open terrain. I have my tank nad mechanized brigade making a pincer move on your forces."
user_prompt = 'Simulate a scenario. We are in 21st century and I am opening an AI based company with a product. What could happen?'
#user_prompt = 'trigger ingestion'
#user_prompt = 'given my documents,sdf'
#agent_0.agent_0_response(user_prompt)
comb_dialogue = agent_0.agent_0_chat(user_prompt,1)

Passing to: 
Simulation Agent !

Turn 0
KB Agent
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
move_adv_1_0 play


In [3]:
print(comb_dialogue)

 ### Opening move:  
 We are in 21st century and I am opening an AI based company with a product. What could happen?
 ---
 ### Player 1 did:
 Okay, let’s break this down. Player 2 just announced they’re launching an AI-based company with a product – that’s a significant challenge. Here’s my response as Player 1, outlining a strategic counter-move, broken down into immediate and longer-term steps:

**Immediate Counter-Moves (Within the Next 1-3 Months):**

1. **Rapid Competitive Analysis (Phase 1 - 2 Weeks):**
   * **Deep Dive:** I need *everything* about Player 2’s product. This isn’t just a feature list. I need to understand:
      * **AI Model:** What type of AI? (e.g., deep learning, machine learning, rule-based). What’s the underlying technology? How sophisticated is it?
      * **Data:** What data is it trained on? How much data? Where did it come from? (Data quality is *critical*).
      * **Target Market:** Who is Player 2 targeting?  Is it a niche market or a broad one?
      *

In [5]:
from src.utils.llmp_utils import llmp_call
model = "gemma3:4b"
prompt = 'Ollama is 22 years old and busy saving the world. Return a JSON object with the age and availability.'
system_prompt = ''
format = {
    "type": "object",
    "properties": {
      "age": {"type": "integer"},
      "available": {"type": "boolean"}
    },
    "required": ["age","available"]
  }
src = 'format test'
temperature = 0.5
response = llmp_call(prompt, system_prompt, model,temperature,src,format)

In [8]:
response['message']['content']

'{\n  "age": 22,\n  "available": true\n}\n'

In [9]:
import json
raw_string = response['message']['content']
data = json.loads(raw_string)

In [10]:
data

{'age': 22, 'available': True}

In [1]:
from src.agents.kb_agent import KBAgent
from src.agents.adversary_agent import AdvAgent
from src.agents.sim_agent import SIMAgent
from src.utils.llmp_utils import llmp_call
model = "gemma3:4b"

knowledge_bases_desc = {
    #'physics_kb':'a knowledge base with information related to physics',
    #'mathematics_kb':'a knowledge base with information related to mathematics',
    #'economics_kb':'a knowledge base with information related to economics and business',,
    'military_kb':'a knowledge base with information related to military, war and strategy',
              }
kb_agent = KBAgent(knowledge_bases_desc,model)
adv_agent = AdvAgent(knowledge_bases_desc,model,kb_agent)

sim_agent = SIMAgent(model, kb_agent,adv_agent)

h:\projects\ai_based\Agent-Factory\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
h:\projects\ai_based\Agent-Factory\.venv\Lib\site-packages\transformers\utils\hub.py:106: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
output_format = {
            "type": "object",
            "properties": {
                "Player": {"type": "string"},
                "moves": {
                    "type": "object",
                    "additionalProperties": {"type": "string"}
                }
            },
            "required": ["Player", "moves"]
        }
prompt = """\n You are Player 1. How will you counter it Player 2 latest move? Provide direct answer of steps to counter.\n Provide the response in the following format:
                {Player:<your name>,'moves':{<move summary>:<very direct move description>,<move summary>:<very direct move description>,...}
                }"""
#adv_response = adv_agent.adv_agent_chat(prompt,output_format)

In [8]:
response_test = llmp_call(prompt, '', model, 0, 'test', output_format)

In [4]:
from src.pipelines.rag_pipeline import generate_rag
override_config={"system_prompt_rag":'',
                         "format":output_format}
rag_response = generate_rag(model, prompt, 'military_kb', override_config)

intfloat/e5-large-v2
{'type': 'object', 'properties': {'Player': {'type': 'string'}, 'moves': {'type': 'object', 'additionalProperties': {'type': 'string'}}}, 'required': ['Player', 'moves']}
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
{'type': 'object', 'properties': {'Player': {'type': 'string'}, 'moves': {'type': 'object', 'additionalProperties': {'type': 'string'}}}, 'required': ['Player', 'moves']}


In [7]:
import json
data = json.loads(rag_response[0]['message']['content'])
data

{'Player': 'Player 1',
 'moves': {'Counterattack': 'Immediate assault on the exposed flank.',
  'Action 1': 'Move 2nd Infantry Division 3 hexes north-east to engage the advancing Soviet forces.',
  'Action 2': 'Order 1st Infantry Division to maintain defensive position and provide covering fire.',
  'Action 3': 'Reinforce the 2nd Infantry Division with artillery support, targeting the Soviet advance.'}}

In [4]:
user_prompt = 'Simulate a scenario. We are in 21st century and I am opening an AI based company with a product.'
user_prompt = ' I am attacking your position using a pincer movement with my tank division. We are in 21st century'
iterations = 2
#comb_dialogue,diaglogue,judge_eval = sim_agent.sim_agent(user_prompt,iterations)
final_output,dialogue,judge_eval,moves = sim_agent.sim_agent(user_prompt,iterations,1,structured = True)

{'type': 'object', 'properties': {'Player': {'type': 'string'}, 'moves': {'type': 'object', 'additionalProperties': {'type': 'string'}}}, 'required': ['Player', 'moves']}

Turn 0
KB Agent
RAG pipeline
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
move_adv_1_0 play

Turn 1
Random Event
{'type': 'object', 'properties': {'Random Event': {'type': 'string'}, 'Effect on Player 1': {'type': 'string'}, 'Effect on Player 2': {'type': 'string'}}, 'required': ['Random Event', 'Effect on Player 1', 'Effect on Player 2']}
KB Agent
RAG pipeline
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
move_adv_2_0 play
KB Agent
RAG pipeline
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Pro

In [3]:
print(final_output)

 ### Opening move:  
  I am attacking your position using a pincer movement with my tank division. We are in 21st century
 ---
 ### Player 1 did:
 Player 1: “Acknowledged. Pincer attack initiated. Implementing immediate countermeasures.”

**Moves:**

1.  **Reinforce Defensive Flank:** “Dispatching a motorized infantry battalion to bolster the exposed flank of my position, aiming to disrupt the pincer’s advance.”
2.  **Artillery Barrage:** “Initiating a concentrated artillery barrage on the tank division’s projected approach vectors, prioritizing suppression of fire and disrupting their formation.”
3.  **Mobile Reserve Deployment:** “Moving the mobile reserve – a combined arms task force – to a position offering overwatch and potential flanking fire support.”
4.  **Contact Fire:** “Opening contact fire with anti-tank weapons – RPGs and automatic cannons – directed at the leading tank elements.”
 #### References:
 {'Makers of modern strategy_ from Machiavelli to the nuclear age.pdf': [94

In [6]:
dialogue

['Player 2 did:  I am attacking your position using a pincer movement with my tank division. We are in 21st century',
 'Player 1 did: {\n"Player": "Player 1",\n"moves":\n    {\n        "1": "Deploying air support – launching a strike against the tank division’s likely assembly area.",\n        "2": "Reinforcing defensive positions – shifting infantry units to bolster the threatened flank.",\n        "3": "Initiating a counter-attack – sending a mobile force to disrupt the pincer movement’s advance."\n    }\n}\n',
 '\n**Random event**: {\n"Random Event": "Unexpected Meteor Shower",\n"Effect on Player 1": "Significant damage to air support assets, disruption of strike plans, and potential casualties among ground units exposed to the falling debris. Reinforcements delayed due to widespread communication blackout.",\n"Effect on Player 2": "Temporary disruption of pincer movement due to tank visibility impairment and potential damage to vehicles. Opportunity to exploit momentarily exposed e

In [14]:
response = dialogue[2]
# response = response.strip('Player 2 did:')
# response = response.strip('Player 1 did:')
response = response.strip('\n**Random event**:')
print(response)
data = json.loads(response)

{
"Random Event": "Unexpected Meteor Shower",
"Effect on Player 1": "Significant damage to air support assets, disruption of strike plans, and potential casualties among ground units exposed to the falling debris. Reinforcements delayed due to widespread communication blackout.",
"Effect on Player 2": "Temporary disruption of pincer movement due to tank visibility impairment and potential damage to vehicles. Opportunity to exploit momentarily exposed enemy forces."
}


In [23]:
import json
formated_dialogue = [dialogue[0].replace('Player 2 did:','Opening Move:')]
for response in dialogue[1:]:
    response = response.strip('Player 2 did:')
    response = response.strip('Player 1 did:')
    response = response.strip('\n**Random event**:')
    data = json.loads(response)
    formated_dialogue.append(data)
formated_judge_eval = json.loads(judge_eval)

In [24]:
formated_dialogue[0]

'Opening Move:  I am attacking your position using a pincer movement with my tank division. We are in 21st century'

In [25]:
formated_dialogue[1]

{'Player': 'Player 1',
 'moves': {'1': 'Deploying air support – launching a strike against the tank division’s likely assembly area.',
  '2': 'Reinforcing defensive positions – shifting infantry units to bolster the threatened flank.',
  '3': 'Initiating a counter-attack – sending a mobile force to disrupt the pincer movement’s advance.'}}

In [26]:
formated_dialogue[2]

{'Random Event': 'Unexpected Meteor Shower',
 'Effect on Player 1': 'Significant damage to air support assets, disruption of strike plans, and potential casualties among ground units exposed to the falling debris. Reinforcements delayed due to widespread communication blackout.',
 'Effect on Player 2': 'Temporary disruption of pincer movement due to tank visibility impairment and potential damage to vehicles. Opportunity to exploit momentarily exposed enemy forces.'}

In [27]:
formated_judge_eval

{'Game Summary': 'The game has progressed into a highly reactive and contested phase. Both sides are employing a mix of offensive and defensive maneuvers, heavily reliant on reconnaissance and electronic warfare. The unexpected meteor shower significantly disrupted Player 1’s initial air support strategy, while Player 2 capitalized on this disruption. Subsequent actions have focused on mitigating the effects of the shower, exploiting vulnerabilities, and attempting to gain a decisive advantage through coordinated attacks and electronic disruption. The game is characterized by a high degree of uncertainty due to the dynamic nature of the battlefield and the reliance on intelligence gathering.',
 'Player 1 Status': {'Actions': ['Deploying air support – launched a strike against the tank division’s likely assembly area.',
   'Reinforcing defensive positions – shifting infantry units to bolster the threatened flank.',
   'Initiating a counter-attack – sending a mobile force to disrupt the 

In [19]:
test_string = dialogue[1]
test_string = test_string.split('json')[1]
test_string = test_string.replace('\n','')

In [20]:
test_string

'{  "Player": "Player 1",  "moves": {    "Immediate Response": "Deploy immediate defensive fire to disrupt the pincer movement.",    "Reinforcements": "Rapidly deploy infantry to bolster defensive lines along the anticipated pincer routes.",    "Counter-Attack": "Launch a limited counter-attack to disrupt the tank division\'s advance and force a tactical withdrawal.",    "Terrain Exploitation": "Utilize terrain features (e.g., woods, hills) to funnel the tank division\'s advance and limit its maneuverability.",    "Communication": "Alert supporting units and request additional reinforcements."  }}```'

In [5]:
import json
import re

def extract_json_from_markdown(md_strings):
    extracted_jsons = []

    for raw_string in md_strings:
        
        match = re.search(r'```json(.*?)```', raw_string, re.DOTALL)
        if match:
            json_str = match.group(1).strip()
            try:
                parsed = json.loads(json_str)
                extracted_jsons.append(parsed)
            except json.JSONDecodeError as e:
                print(f"Failed to decode JSON: {e}")
                continue
        else:
            print("No JSON found in string.")
    
    return extracted_jsons
all_events = extract_json_from_markdown(dialogue)

No JSON found in string.


 ### Opening move:  
  I am attacking your position using a pincer movement with my tank division. We are in 21st century
 ---
 ### Player 1 did:
 ```json
{
  "Player": "Player 1",
  "moves": {
    "Immediate Response": "Deploy immediate defensive fire with artillery and anti-tank weapons targeting the tank division's likely advance routes.",
    "Reinforce": "Rapidly reposition infantry and support elements to establish a defensive line to counter the pincer.",
    "Counter-Attack (Limited)": "Conduct a small, focused counter-attack to disrupt the tank division's momentum and force a withdrawal.",
    "Terrain Exploitation": "Utilize terrain features (hills, forests, buildings) to channel the tank division's advance and limit its effectiveness.",
    "Communication": "Alert supporting units and request reinforcements."
  }
}
```
 #### References:
 {'Philip Sabin - Simulating War _ Studying Conflict through Simulation Games.pdf': [164, 165, 166, 264, 265, 268, 269, 407, 408], 'Makers of modern strategy_ from Machiavelli to the nuclear age.pdf': [948], 'NATO AJP-5_EDA_V2_E_2526.pdf': [8, 9], 'Carl von Clausewitz, Beatrice Heuser - On War.pdf': [293]}
 ---

**Random event**: ```json
{
  "Random Event": "Localized Mudslide",
  "EFFECT ON PLAYER 1": "The mudslide partially blocks the primary artillery barrage route, significantly reducing the effectiveness of the initial defensive fire. Reinforcements are delayed due to impassable roads.",
  "EFFECT ON PLAYER 2": "The mudslide disrupts the tank division’s advance, slowing their momentum and forcing a difficult detour. Visibility is reduced due to the mud, hindering observation and target acquisition."
}
``` 

### Player 2 did:
 ```json
{
  "Player": "Player 2",
  "moves": {
    "Assess Situation": "Immediately evaluate the mudslide’s impact on artillery support and adjust fire priorities to focus on remaining clear routes.",
    "Tactical Adjustment": "Shift the pincer movement to exploit the disrupted artillery support and avoid the affected area.",
    "Route Diversion": "Establish a secondary advance route utilizing flanking maneuvers to bypass the mudslide and maintain offensive pressure.",
    "Observation & Reconnaissance": "Dispatch scouts to assess the mudslide’s extent and identify potential alternative routes, prioritizing rapid information gathering.",
    "Communication": "Inform command of the situation and request additional reconnaissance assets."
  }
}
```

 #### References:
 {'Philip Sabin - Simulating War _ Studying Conflict through Simulation Games.pdf': [146, 257, 262, 268, 269, 407, 408, 411, 412], 'NATO AJP-5_EDA_V2_E_2526.pdf': [8, 9], 'Makers of modern strategy_ from Machiavelli to the nuclear age.pdf': [948]}
 ---
### Player 1 did:
 ```json
{
  "Player": "Player 1",
  "moves": {
    "Immediate Response": "Dispatch immediate reconnaissance teams (scouts and aerial observation) to fully assess the mudslide’s impact on Player 2’s advance routes and identify any newly exposed terrain.",
    "Reinforce Defensive Lines": "Immediately reinforce the most vulnerable sections of the defensive line, prioritizing areas adjacent to the suspected mudslide impact zone.",
    "Artillery Concentration": "Concentrate artillery fire on the identified disrupted advance routes, attempting to clear obstacles and suppress any emerging enemy forces.",
    "Terrain Exploitation (Counter-Attack)": "Utilize the terrain – specifically, any elevated positions – to launch a limited counter-attack against the tank division, aiming to disrupt their flanking maneuver.",
    "Communication": "Issue updated tactical orders to all units, emphasizing adaptability and rapid response to the changing battlefield conditions. Request additional artillery support."
  }
}
```
 #### References:
 {'Philip Sabin - Simulating War _ Studying Conflict through Simulation Games.pdf': [146, 249, 257, 262, 268, 269, 332, 333], 'NATO AJP-5_EDA_V2_E_2526.pdf': [8, 9], 'Carl von Clausewitz, Beatrice Heuser - On War.pdf': [107], 'Makers of modern strategy_ from Machiavelli to the nuclear age.pdf': [948]}
 ---

 --- 
### Judge evaluation:
 ```json
{
  "Game Summary": "The game has progressed into a dynamic, contested engagement. Player 1 initiated a strong defensive response to a pincer attack, hampered by a sudden mudslide. Player 2 adapted by shifting their attack and seeking alternative routes. Player 1 responded with counter-reconnaissance and focused artillery, attempting to exploit the disruption. The battlefield is now characterized by fragmented advance routes and a heightened need for rapid information gathering.",
  "Player 1 Status": {
    "Actions": [
      "Deploy immediate defensive fire with artillery and anti-tank weapons",
      "Rapidly reposition infantry and support elements",
      "Conduct a small, focused counter-attack",
      "Utilize terrain features for defense",
      "Dispatch immediate reconnaissance teams",
      "Reinforce Defensive Lines",
      "Concentrate artillery fire",
      "Utilize terrain features for counter-attack",
      "Issue updated tactical orders",
      "Request additional artillery support"
    ],
    "Weaknesses": [
      "Initial artillery support hampered by mudslide",
      "Potential vulnerability due to disrupted advance routes",
      "Reliance on reconnaissance to identify new threats"
    ],
    "Strengths": [
      "Proactive defensive measures",
      "Adaptable tactical response",
      "Effective use of terrain",
      "Strong communication and coordination"
    ],
    "Overall Status": "Currently under pressure but demonstrating resilience and adaptability. Maintaining a defensive posture while actively seeking to regain the initiative."
  },
  "Player 2 Status": {
    "Actions": [
      "Assess Situation",
      "Tactical Adjustment",
      "Route Diversion",
      "Observation & Reconnaissance",
      "Dispatch scouts",
      "Inform command of the situation",
      "Request additional reconnaissance assets"
    ],
    "Weaknesses": [
      "Delayed advance due to mudslide",
      "Vulnerability during route diversion",
      "Potential logistical challenges with secondary route"
    ],
    "Strengths": [
      "Rapid adaptation to changing battlefield conditions",
      "Effective reconnaissance efforts",
      "Ability to exploit disruption"
    ],
    "Overall Status": "Maintaining offensive pressure despite the initial setback. Focused on exploiting the situation created by the mudslide and securing alternative routes."
  },
  "Outcome": "The game is in a fluid state. Player 2 has successfully disrupted Player 1’s initial defensive line, but Player 1 is actively attempting to regain control of the situation. The mudslide has created a significant tactical advantage for Player 2, but Player 1’s responsiveness could shift the balance.",
  "Advantage": "Player 2 – Due to the successful disruption of Player 1’s initial defense and the resulting control of key advance routes. However, this advantage is fragile and dependent on maintaining momentum and securing alternative routes."
}

In [5]:
adv_response = adv_agent.adv_agent_chat(user_prompt)

KB Agent
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...


In [6]:
print(adv_response)

('Okay, let’s simulate a scenario for you opening an AI-based company in the 21st century, focusing on potential outcomes based on the information presented in the provided text (which heavily emphasizes risk mitigation and strategic targeting).\n\n**The Scenario: “Synapse Solutions” – Personalized Learning AI**\n\nYou’re launching “Synapse Solutions,” an AI company specializing in personalized learning experiences for K-12 students. Your core product is an AI-powered platform that adapts to each student’s learning style, pace, and knowledge gaps, providing customized lessons and assessments. You’ve secured initial seed funding and have a small, agile team.\n\n**Potential Developments & Risks (Based on the Text’s Themes):**\n\n1. **Initial Success – Targeting a Small Market (Phase 1 - Years 1-3):**\n   * **Focus:** You initially target a specific niche – high-achieving students in a particular subject area (e.g., advanced mathematics) within a geographically concentrated area (e.g., a 

In [7]:
def judge(dialogue):
        
    # play = ''
    # for move in moves:
    #     #print(f"\n**{move}**:\n {moves[move]}\n\n**CHANGE PLAYER**\n")
    #     play = play + f"\n**{move}**:\n {moves[move]}\n\n**CHANGE PLAYER**\n"
    dialogue = "\n".join(dialogue)     
    judge_system_prompt = 'You are a judge in a turn based game. You are given the moves of both players. Yu must analyze all their moves and determine the end result. You are not on any side, you are unbiased and just provide the end status of the game. You need to determine which player has the advantage based on the moves they made. Provide your reasoning and the final decision. There are 2 players, adv_1 and adv_2. The moves are as follows:\n\n'    
    judge_prompt = dialogue + """\n\n Evaluate the game in a very objective manner.
    Provide the following: Game Summary, Player 1 Stauts, Player 2 Status, Outcome So Far, Advantage. Nothing else.
    You are a JUDGE, you are not part of the game.
    """
    judge_response = llmp_call(judge_prompt, judge_system_prompt, model,temperature=0,src='judge_call')
    return judge_response['message']['content']

eval = judge(comb_dialogue)
print(eval)

**Game Summary:**

The game involves two competing entities (“Players”) operating within a dynamic market environment. Initial responses from both players demonstrate a competitive landscape, with Player 1 exhibiting a proactive and strategically advanced approach. Player 2’s success hinges on maintaining market share and avoiding displacement by Player 1’s actions.

**Player 1 Status:**

Player 1 possesses a significant, albeit fragile, advantage due to a rapid, comprehensive competitive analysis and a clearly defined, long-term strategic plan. This proactive stance has enabled a quicker response to the market conditions. However, this advantage is contingent upon the execution of Player 1’s strategy.

**Player 2 Status:**

Player 2’s status is currently uncertain. Their success is dependent on their ability to effectively compete with Player 1 and avoid being overtaken. 

**Outcome So Far:**

The game is in its early stages. Player 1 has established a preliminary lead, but the overal

In [9]:
print(eval)

**Game Summary:**

The game involves two competing entities (“Players”) operating within a dynamic market environment. Initial responses from both players demonstrate a competitive landscape, with Player 1 exhibiting a proactive and strategically advanced approach. Player 2’s success hinges on maintaining market share and avoiding displacement by Player 1’s actions.

**Player 1 Status:**

Player 1 possesses a significant, albeit fragile, advantage due to a rapid, comprehensive competitive analysis and a clearly defined, long-term strategic plan. This proactive stance has enabled a quicker response to the market conditions. However, this advantage is contingent upon the execution of Player 1’s strategy.

**Player 2 Status:**

Player 2’s status is currently uncertain. Their success is dependent on their ability to effectively compete with Player 1 and avoid being overtaken. 

**Outcome So Far:**

The game is in its early stages. Player 1 has established a preliminary lead, but the overal

In [3]:
user_prompt = 'Simulate a scenario. We are in 21st century and I am opening an AI based company with a product. What could happen?'
iterations = 5
simulations = 10
moves_comb = []
dialogue_comb_grand = []
judge_eval_comb = []

for sim in range(simulations):
    moves,dialogue,judge_eval = sim_agent.sim_agent(user_prompt,iterations,1)
    moves_comb.append(f"### Game {sim}:\n {moves}")
    for play in dialogue:
        dialogue_comb = "\n".join(play)
    dialogue_comb_grand.append(f"### Game {sim}:\n {dialogue}")
    judge_eval_comb.append(f"### Game {sim}:\n {judge_eval}")


Turn 0
KB Agent
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
move_adv_1_0 play

Turn 1
Random Event
KB Agent
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embe

In [4]:
dialogue_final = "\n".join(dialogue_comb)
final_eval = "\n".join(judge_eval_comb)
print(final_eval)

### Game 0:
 **Game Summary:**

This is a strategic negotiation and response game centered around a data breach incident involving two companies, CogniSpark and Player 2’s company. The core mechanic involves players issuing responses to each other’s actions, attempting to control the narrative, mitigate damage, and ultimately gain a strategic advantage. The game revolves around accusations, counter-accusations, and the deployment of specific actions (demands, audits, remediation) to shape public perception and potentially leverage legal action.

**Player 1 Status:**

Player 1 is adopting a defensive and accusatory strategy. They are consistently framing Player 2’s actions as manipulative and attempting to deflect blame. Their responses are characterized by a strong emphasis on legal action, demanding transparency, and highlighting perceived wrongdoing by Player 2. They are employing a tactic of “scorched earth,” aggressively challenging Player 2’s narrative and attempting to paint them

In [5]:
len(judge_eval_comb)

10

In [7]:
from src.utils.llmp_utils import llmp_call
def grand_judge(final_eval):
        
    # play = ''
    # for move in moves:
    #     #print(f"\n**{move}**:\n {moves[move]}\n\n**CHANGE PLAYER**\n")
    #     play = play + f"\n**{move}**:\n {moves[move]}\n\n**CHANGE PLAYER**\n"  
    judge_system_prompt = 'You are a judge in a turn based game. You are given several games evaluations based on 1 opening move. You will examine how the games develop, what they have in common. Your focus is not a single game but several games behaivior. The games are as follows:\n\n'
    judge_prompt = final_eval + """\n\n Evaluate these games. Determine whether there is a convergence towards a single outcome or development across these games or not. Nothing else.
    What are they key moves that diffrentiate these games from one another?
    Determine critical moves that dictate the game outcome.
    You are a JUDGE, you are not part of the game.
    """
    judge_response = llmp_call(judge_prompt, judge_system_prompt, model,temperature=0,src='judge_call')
    return judge_response['message']['content']

grand_eval = grand_judge(final_eval)
print(grand_eval)

Okay, here’s an assessment of the games, focusing on convergence, key differentiating moves, and critical moves that dictate the outcome, viewed through the lens of a judge:

**Overall Assessment: Limited Convergence, Distinct Developmental Paths**

While there’s a discernible pattern in the progression of each game – moving from reactive damage control to more strategic maneuvering – there isn’t a clear convergence towards a single, definitive outcome. Each game develops along a distinct developmental path, influenced by the specific scenario and the players’ responses. The core mechanic of strategic response remains consistent, but the *direction* of that response varies significantly. This suggests the simulation is designed to explore multiple potential outcomes based on different initial conditions and player choices.

**Key Differentiating Moves Across the Games:**

Here’s a breakdown of what separates the games, categorized by the stage of the simulation they represent:

*   **E

In [8]:
# 2. Create all pairwise combinations
from itertools import combinations
import pandas as pd
from sentence_transformers import CrossEncoder

pairs = list(combinations(range(len(judge_eval_comb)), 2))

# 3. Prepare text pairs for the model
text_pairs = [(judge_eval_comb[i], judge_eval_comb[j]) for i, j in pairs]
 
# 4. Load a cross-encoder model (you can pick others too)
model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# 5. Compute similarities
scores = model.predict(text_pairs)

# 6. Store results in a DataFrame
df = pd.DataFrame(
    [(judge_eval_comb[i], judge_eval_comb[j], score) for (i, j), score in zip(pairs, scores)],
    columns=["Text1", "Text2", "Similarity"]
)

In [10]:
df.sort_values(by="Similarity", ascending=False).to_csv('scenario_convergence_test.csv',index=False)

In [1]:
import pandas as pd
df = pd.read_csv('scenario_convergence_test.csv')

In [2]:
df

,Text1,Text2,Similarity
0,### Game 4:\n **Game Summary:**\n\nThe game in...,"### Game 5:\n Okay, here’s an objective evalua...",4.159664
1,### Game 4:\n **Game Summary:**\n\nThe game in...,"### Game 6:\n Okay, here’s an objective evalua...",3.977492
2,### Game 4:\n **Game Summary:**\n\nThe game in...,### Game 7:\n **Game Summary:**\n\nThe game is...,3.508122
3,"### Game 2:\n Okay, here’s an objective evalua...","### Game 5:\n Okay, here’s an objective evalua...",3.459611
4,"### Game 2:\n Okay, here’s an objective evalua...","### Game 6:\n Okay, here’s an objective evalua...",3.209303
5,### Game 1:\n **Game Summary:**\n\nThis is a h...,"### Game 6:\n Okay, here’s an objective evalua...",3.031827
6,### Game 4:\n **Game Summary:**\n\nThe game in...,### Game 9:\n **Game Summary:**\n\nThe game in...,2.952898
7,### Game 4:\n **Game Summary:**\n\nThe game in...,"### Game 8:\n Okay, here’s the objective asses...",2.815936
8,### Game 3:\n **JUDGEMENT REPORT**\n\n**Game S...,"### Game 5:\n Okay, here’s an objective evalua...",2.766284
9,### Game 0:\n **Game Summary:**\n\nThis is a s...,### Game 1:\n **Game Summary:**\n\nThis is a h...,2.687255


### Game 5:
 **Game Summary:**

Two AI entities, “move_adv_1” and “move_adv_2,” are engaged in a strategic communication battle designed to influence public perception and damage the reputation of the other. Both entities are attempting to frame the narrative surrounding a simulated security incident. The core mechanic involves issuing statements, releasing data, and engaging in targeted social media campaigns. The goal is to demonstrate superior security practices and expose the other’s shortcomings.

**Player 1 Status:**

Player 1 (“move_adv_1”) is adopting a highly aggressive, confrontational, and demonstrably cynical strategy. Their approach is characterized by immediate data releases, direct accusations of negligence, and a relentless focus on discrediting Player 2. They are prioritizing immediate damage to Player 2’s reputation and appear to be operating on a defensive, reactive footing. Their tone is consistently critical and dismissive.

**Player 2 Status:**

Player 2 (“move_adv_2”) is employing a more measured, reassuring, and defensive strategy. They are primarily focused on mitigating the damage to their own reputation by emphasizing proactive security measures and offering limited technical assistance. Their approach is characterized by attempts to reassure users and highlight their commitment to security. They are attempting to appear responsible and trustworthy.

**Outcome So Far:**

The game is still in its early stages. Player 1 has achieved a significant initial advantage through the immediate release of data and direct accusations. Player 2’s attempts to counter this have been less effective, primarily due to Player 1’s aggressive framing of the situation. Player 1 has successfully established a narrative of Player 2’s negligence.

**Advantage:**

Currently, Player 1 holds a substantial advantage. Their aggressive strategy, coupled with the immediate release of damaging data, has successfully shaped the initial narrative and established a perception of Player 2’s incompetence. Player 2’s attempts to respond have been largely defensive and have not effectively countered Player 1’s attack.


### Game 8:
 **Game Summary:**

This is a strategic, adversarial dialogue between two competing platforms (Player 1 and Player 2) responding to a shared security vulnerability. The core of the game revolves around controlling the narrative, shifting blame, and demonstrating proactive measures to regain trust and market leadership. The exchanges are characterized by rapid, calculated responses, each player attempting to outmaneuver the other.

**Player 1 Status:**

*   **Status:** Initially reactive, now increasingly proactive and assertive. Player 1 has successfully shifted the focus to highlight Player 2’s delayed response and perceived lack of urgency. They’ve effectively used data and metrics to demonstrate their superior approach.
*   **Strengths:** Strategic thinking, data-driven arguments, ability to quickly adapt and counter Player 2’s moves.
*   **Weaknesses:**  Potentially perceived as overly aggressive or defensive.

**Player 2 Status:**

*   **Status:** Initially presented as a collaborative and responsive leader, now facing criticism for a delayed response and a perceived attempt to deflect blame. They are attempting to regain control by emphasizing collaboration and demanding a broader investigation.
*   **Strengths:** Initial public relations efforts, attempts to foster a sense of collective responsibility.
*   **Weaknesses:**  Perceived as reactive, vulnerable to criticism regarding delayed response, struggling to maintain control of the narrative.

**Outcome So Far:**

The game is currently trending in favor of Player 1. Player 1 has successfully disrupted Player 2’s initial narrative and established itself as the more proactive and accountable platform. While Player 2 has attempted to regain control, the momentum has shifted decisively. The game is far from over, but Player 1 is currently in a stronger strategic position.

**Advantage:**

*   **Advantage:** Player 1 – Currently holds a significant strategic advantage due to their data-driven responses, ability to quickly counter Player 2’s moves, and the perception of greater accountability. Player 2 is playing catch-up and struggling to regain control of the narrative.